# Session 11: Advanced Retrieval with LangChain

## Learning Objectives:

- Understand and implement multiple retrieval strategies for RAG
- Compare naive, BM25, multi-query, parent-document, contextual compression, ensemble, and semantic chunking approaches
- Build RAG chains over a health and wellness knowledge base using LangChain and QDrant

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

---

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

> NOTE: Create a `.env` file in this directory with `OPENAI_API_KEY` and `COHERE_API_KEY` to avoid being prompted each time.

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Health and Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, stress management, habits, and common health concerns.

### Data Preparation

We'll load the wellness guide as a single document, then split it into smaller chunks using a `RecursiveCharacterTextSplitter` for our vector store. We also keep the raw (unsplit) document for use with the Parent Document Retriever and Semantic Chunker later.

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/HealthWellnessGuide.txt")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
wellness_docs = text_splitter.split_documents(raw_docs)

Let's verify our data was loaded and split correctly!

In [4]:
print(f"Raw documents: {len(raw_docs)}")
print(f"Split chunks: {len(wellness_docs)}")
print(f"\nExample chunk:\n{wellness_docs[0]}")

Raw documents: 1
Split chunks: 45

Example chunk:
page_content='The Personal Wellness Guide
A Comprehensive Resource for Health and Well-being

PART 1: EXERCISE AND MOVEMENT

Chapter 1: Understanding Exercise Basics

Exercise is one of the most important things you can do for your health. Regular physical activity can improve your brain health, help manage weight, reduce the risk of disease, strengthen bones and muscles, and improve your ability to do everyday activities.' metadata={'source': 'data/HealthWellnessGuide.txt'}


## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "wellness_guide".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [5]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = QdrantVectorStore.from_documents(
    wellness_docs,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide",
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [6]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [8]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [9]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [10]:
naive_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help alleviate lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold each for 5 seconds, then switch sides. Perform 10 repetitions per side.\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis slightly. Hold for 10 seconds and repeat 8-12 times.\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over chest, tighten stomach muscles, and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\nThese gentle exercises can help strengthen and stretch the muscles s

In [11]:
naive_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. Adequate and quality sleep, typically 7-9 hours per night for adults, supports physical health by allowing the body to repair tissues, regenerate, and regulate hormones involved in growth and appetite. It also enhances mental well-being, including memory consolidation and cognitive function. Additionally, good sleep strengthens the immune system, helping the body to fight off illnesses, and contributes to emotional stability. Poor sleep or insomnia can lead to health issues such as increased stress, compromised immune function, and higher risk of chronic conditions. Therefore, maintaining healthy sleep habits and environment is essential for overall wellness.'

In [12]:
naive_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking plenty of water to stay hydrated\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gently massaging the temples and neck\n- Using essential oils such as peppermint or lavender\n- Maintaining a regular sleep schedule\n- Practicing deep breathing, progressive muscle relaxation, or grounding techniques for immediate stress relief\n- Taking short walks, especially in nature\n- Listening to calming music\n\nThese approaches can help alleviate headaches and reduce stress naturally.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [13]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(wellness_docs)

We'll construct the same chain - only changing the retriever.

In [14]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [15]:
bm25_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Aim for 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold each extension for about 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Pelvic Tilts:** Lie on your back with knees bent, tighten your abs and tilt your pelvis upward to flatten your back against the floor. Hold for 10 seconds and repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises can help alleviate lower back discomfort and may prevent future issues.'

In [16]:
bm25_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. Maintaining a regular sleep schedule and creating a comfortable sleep environment are essential for promoting quality sleep. Good sleep hygiene practices—such as keeping the bedroom cool, dark, and quiet; avoiding screens before bed; and limiting caffeine and heavy meals at night—help improve sleep consistency and quality. Adequate rest supports various aspects of health, including immune function, mental well-being, and optimal digestive health. Poor sleep or insomnia can negatively affect these areas, highlighting the importance of good sleep habits for overall wellness.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include relaxation techniques such as progressive muscle relaxation, meditation, and deep breathing exercises. Herbal teas like chamomile or valerian root may also help promote relaxation. Additionally, staying well-hydrated, managing stress through mindfulness or exercise, and ensuring adequate sleep can be beneficial in reducing headaches related to stress.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### Answer:

BM25 can perform better when queries are looking for exact lexical matches. In that case, we don't want semantic results, and semantic-only retrievers may miss the match entirely. A good example would be if someone was looking for a folder called "Marketing Materials 2025" and they searched "marketing materials". They don't want matches for "Promotional materials", "Marketing documents", or "Advertising content" even though a semantic search system might retrieve those extra items. 

BM25 can do better with content that is specialized and where searches include acronyms, proper nouns, or specialized concepts that common embedding models aren't trained on (this can be helped with fine-tuning, though).

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [18]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [19]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [20]:
contextual_compression_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Based on the provided information, several exercises can help with lower back pain:\n\n1. **Cat-Cow Stretch**: Begin on hands and knees. Arch your back upwards (like a cat) then gently sag it down (like a cow). Repeat 10-15 times.\n\n2. **Bird Dog**: On hands and knees, extend opposite arm and leg while engaging your core. Hold for about 5 seconds, then switch sides. Aim for 10 repetitions per side.\n\n3. **Pelvic Tilts**: Lie on your back with knees bent. Flatten your back against the floor by tightening your abdominal muscles and tilting your pelvis slightly. Hold for 10 seconds, then repeat 8-12 times.\n\nThese exercises can help alleviate discomfort and strengthen the muscles supporting your lower back. However, it’s best to consult with a healthcare professional before starting any new exercise routine, especially if you have ongoing pain or other health concerns.'

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep is essential for overall health. It supports physical recovery by allowing the body to repair tissues and regenerate cells. Sleep also plays a vital role in mental well-being, helping to consolidate memories and regulate hormones that influence growth and appetite. Additionally, adequate sleep promotes cognitive function and emotional stability. Disrupted or insufficient sleep can negatively impact health, so maintaining good sleep habits and creating an optimal sleep environment are important for overall wellness.'

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include deep breathing exercises, progressive muscle relaxation, grounding techniques, taking a short walk in nature, listening to calming music, staying well-hydrated by drinking water, resting in a dark and quiet room, gentle massage of the temples and neck, using peppermint or lavender essential oils, and maintaining a regular sleep schedule.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [23]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [24]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [25]:
multi_query_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles, and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n\nThese exercises are gentle stretching and strengthening movements that can help alleviate lower back discomfort and pr

In [26]:
multi_query_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical repair, mental well-being, and cognitive functions. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep—typically 7-9 hours per night—is associated with a stronger immune system, better mental health, and a greater capacity for learning and memory. Conversely, poor sleep or sleep disturbances like insomnia can negatively impact your health, increasing the risk of issues such as stress, fatigue, and other health problems. Therefore, maintaining good sleep hygiene and routines is essential for sustaining overall health and well-being.'

In [27]:
multi_query_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking water to stay hydrated\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of the temples and neck\n- Using essential oils such as peppermint or lavender\n- Taking small amounts of caffeine (though it can help or hurt)\n- Performing deep breathing exercises (inhale for 4 counts, hold for 4, exhale for 4)\n- Practicing progressive muscle relaxation, tensing and releasing muscle groups\n- Engaging in grounding techniques, such as naming things you see, hear, feel, smell, and taste\n- Going for a short walk, ideally in nature\n- Listening to calming music\n\nAdditionally, maintaining regular sleep routines and managing stress through activities like mindfulness, meditation, regular exercise, and hobbies can also help reduce headaches and stress.'

### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### Answer:
Multiple reformulations of a query can catch related chunks that might not have been retrieved based on the initial user query. Reformulations of the same query can capture different embeddings in vector space and match different documents. Essentially, we're giving ourselves several different "flavors" of the same question, hoping to increase the retrieved content while still keeping it conceptually related to the initial question.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. We split the full document into large "parent" chunks (e.g. 2000 characters).
2. Each parent chunk is further split into smaller "child" chunks (e.g. 400 characters).
3. The child chunks are stored in a VectorStore, while the parent chunks are stored in an in-memory docstore.
4. When we query our Retriever, we do a similarity search comparing our query vector to the child chunks.
5. Instead of returning the child chunks, we return their associated parent chunks.

The basic idea is:

- **Search** for small, focused chunks (better semantic matching)
- **Return** big chunks (richer surrounding context)

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by defining our parent and child splitters.

In [28]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [29]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="wellness_parent_child",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="wellness_parent_child", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [30]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore=parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [31]:
parent_document_retriever.add_documents(raw_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [32]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [33]:
parent_document_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

"For lower back pain, several gentle stretching and strengthening exercises can help alleviate discomfort and prevent future episodes. These include:\n\n- Cat-Cow Stretch: On hands and knees, alternate arching your back up (cat) and sagging it down (cow), doing 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg, hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten your stomach muscles, and lift your shoulders off the floor. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: While lying on your back, pull one knee toward your chest, hold for 15-30 seconds, then switch legs.\n- Pelvic Tilts: Lie on your back with knees bent, tilt your pelvis up and flatten your back against the floor, hold for 10 seconds. Repeat 8-12 times.\n\nThese exercises can help strengthen the muscles supporting your lower back and improve flexibility. However, it's always best to consult w

In [34]:
parent_document_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep significantly affects overall health in several key ways. It is essential for physical health, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adults generally need 7-9 hours of sleep per night, and proper sleep cycles (about 90 minutes each, alternating between REM and non-REM sleep) are important for these processes. Good sleep hygiene practices—such as maintaining a consistent schedule, creating a restful environment, and limiting screen exposure before bed—help promote quality sleep. Adequate sleep supports energy levels, mood, immune function, and overall physical and mental health.'

In [35]:
parent_document_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Deep breathing exercises, such as inhaling for 4 counts, holding, and exhaling for 4 counts.\n- Progressive muscle relaxation, which involves tensing and releasing muscle groups throughout the body.\n- Mindfulness and meditation practices to promote present-moment awareness and reduce stress.\n- Gentle massage of the temples and neck to relieve tension.\n- Using essential oils like peppermint or lavender, which can have soothing effects.\n- Engaging in regular physical activity or hobbies to help manage stress levels.\n- Maintaining good sleep hygiene, such as having a consistent sleep schedule and creating a calming bedtime routine.\n- Staying well-hydrated by drinking plenty of water, as dehydration can trigger headaches.\n- Resting in a dark, quiet room during headache episodes.\n- Applying warm or cold compresses to the head or neck to alleviate headache pain.\n\nAlways consult with a healthcare professional if headaches

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [36]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

Let's look at our results!

In [38]:
ensemble_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Pelvic Tilts:** Lie on your back with knees bent, tighten your abs, and tilt your pelvis up slightly to flatten your back against the floor. Hold for 10 seconds and repeat 8-12 times.\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over chest, tighten stomach muscles, and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\nThese gentle stretching and strengthening exercises can help alleviate discomfort and prev

In [39]:
ensemble_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical recovery, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep, typically 7-9 hours per night, is essential for maintaining immune function, managing stress, and promoting emotional stability. Poor sleep or sleep disturbances like insomnia can negatively impact health, increasing the risk for chronic conditions and reducing quality of life. Therefore, prioritizing good sleep practices and creating an optimal sleep environment are important steps to enhance overall health.'

In [40]:
ensemble_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Deep breathing exercises, such as inhaling for 4 counts, holding for 4, and exhaling for 4.\n- Progressive muscle relaxation, tensing and releasing muscle groups from toes to head.\n- Grounding techniques, like noticing 5 things you see, 4 you hear, 3 you feel, 2 you smell, and 1 you taste.\n- Taking short walks, preferably in nature.\n- Listening to calming music.\n- Drinking water to stay hydrated.\n- Applying cold or warm compresses to the head or neck.\n- Resting in a dark, quiet room.\n- Gentle massage of temples and neck.\n- Using essential oils like peppermint or lavender.\n- Maintaining a regular sleep schedule.\n  \nThese approaches can help alleviate headaches and reduce stress naturally.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [41]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [42]:
semantic_documents = semantic_chunker.split_documents(raw_docs)

Let's create a new vector store.

In [43]:
semantic_vectorstore = QdrantVectorStore.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide_semantic_chunks"
)

We'll use naive retrieval for this example.

In [44]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [45]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [47]:
semantic_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

"Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over chest, tighten stomach muscles, and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n\nThese exercises are gentle and aimed at alleviating discomfort and preventing future episodes of lower back pain. However, it's always best to consult with a healthcare professional before starting any new exercise routine, especially if you have sp

In [48]:
semantic_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical, mental, and cognitive well-being. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep (generally 7-9 hours for adults) helps maintain a healthy immune system, supports emotional stability, improves concentration, and reduces the risk of chronic diseases. Poor sleep or sleep disturbances can lead to fatigue, stress, impaired immune function, mood issues, and increased risk for health problems. Therefore, quality sleep is essential for maintaining optimal overall health.'

In [49]:
semantic_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- For Stress:\n  - Deep breathing exercises (e.g., inhale for 4 counts, hold for 4, exhale for 4)\n  - Progressive muscle relaxation\n  - Grounding techniques (e.g., naming things you see, hear, feel, smell, taste)\n  - Taking a short walk, preferably in nature\n  - Listening to calming music\n  - Practicing mindfulness and meditation (focused attention, body scan, loving-kindness, walking meditation, guided meditation)\n  \n- For Headaches:\n  - Drinking plenty of water to stay hydrated\n  - Applying cold or warm compresses to the head or neck\n  - Resting in a dark, quiet room\n  - Gentle massage of temples and neck\n  - Using peppermint or lavender essential oils\n  - Consuming small amounts of caffeine (with caution)\n  - Maintaining a regular sleep schedule\n\nThese remedies leverage natural techniques and substances to help alleviate stress and headaches.'

### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### Answer:

In semantic chunking, we are embedding each sentence and then grouping the sentences based on semantic similarity. We choose chunk breaks where similarity falls under a certain threshold. With documents where sentences are short and highly repetitive, semantic chunking might lead to very few giant chunks. We would have to play with the chunking thresholds to be more attuned to smaller differences to build good chunks. Honestly, I might consider another chunking method if the entire doc is semantically related - chunking based on structure might be a good choice for FAQs.

---

# 🤝 Breakout Room Part #2

### 🏗️ Activity #1:

Your task is to evaluate the various Retriever methods against each other.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparison between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [56]:
### YOUR CODE HERE

# Create the "golden dataset"
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from ragas.testset import TestsetGenerator

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(embeddings)

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(raw_docs, testset_size=10)



/var/folders/vv/l_xmncsj59gbr7ptc7h1ghvm0000gn/T/ipykernel_67708/3968633771.py:9: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
/var/folders/vv/l_xmncsj59gbr7ptc7h1ghvm0000gn/T/ipykernel_67708/3968633771.py:10: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(embeddings)


Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/5 [00:00<?, ?it/s]

...

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Skipping multi_hop_abstract_query_synthesizer due to unexpected error: No relationships match the provided condition. Cannot form clusters.


Generating personas:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [62]:
dataset.to_pandas()

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,What exercises are recommended for relieving l...,[PART 1: EXERCISE AND MOVEMENT\n\nChapter 1: U...,Recommended exercises for lower back pain incl...,Wellness Enthusiast,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
1,wut is burd dog exersize?,[PART 1: EXERCISE AND MOVEMENT\n\nChapter 1: U...,Bird Dog is an exercise where you start on han...,Wellness Enthusiast,MISSPELLED,SHORT,single_hop_specific_query_synthesizer
2,As someone dedicated to maintaining a balanced...,[PART 1: EXERCISE AND MOVEMENT\n\nChapter 1: U...,To properly perform Pelvic Tilts for lower bac...,Wellness Enthusiast,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer
3,Why proteins matter for me health and what foo...,[PART 2: NUTRITION AND DIET\n\nChapter 4: Fund...,Proteins is essential for muscle repair and im...,Wellness Enthusiast,POOR_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
4,How can magnesium be incorporated into a welln...,[PART 2: NUTRITION AND DIET\n\nChapter 4: Fund...,"According to the provided context, magnesium s...",Wellness Enthusiast,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
5,What is Cognitive Behavioral Therapy for Insom...,[PART 2: NUTRITION AND DIET\n\nChapter 4: Fund...,Cognitive Behavioral Therapy for Insomnia (CBT...,Wellness Enthusiast,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
6,What Chapter 15 say about evening wind-down ro...,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,Chapter 15 explains that a consistent evening ...,Wellness Enthusiast,POOR_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
7,wat is chaptre 13 about habbits?,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,Chapter 13 explains that habits are behaviors ...,Wellness Enthusiast,MISSPELLED,SHORT,single_hop_specific_query_synthesizer
8,wut is PART 6 about in this book?,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,PART 6 is titled COMMON HEALTH CONCERNS and co...,Wellness Enthusiast,MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer
9,wat is walkin meditaton?,[PART 4: STRESS MANAGEMENT AND MENTAL WELLNESS...,Walking meditation is a type of meditation tha...,Wellness Enthusiast,MISSPELLED,SHORT,single_hop_specific_query_synthesizer


In [64]:
from ragas import EvaluationDataset
import time

results = {}

retriever_chains = {
    "naive": naive_retrieval_chain,
    "bm25": bm25_retrieval_chain,
    "parent_document": parent_document_retrieval_chain,
    "compression": contextual_compression_retrieval_chain,
    "multi_query": multi_query_retrieval_chain,
    "ensemble": ensemble_retrieval_chain,
}

for name, chain in retriever_chains.items():
    eval_samples = []

    for test_row in dataset:
        response = chain.invoke(
            {"question": test_row.eval_sample.user_input},
            config={"callbacks": [], "run_name": name, "project_name": f"session11-{name}"}
        )
        eval_samples.append({
            "user_input": test_row.eval_sample.user_input,
            "reference": test_row.eval_sample.reference,
            "reference_contexts": test_row.eval_sample.reference_contexts,
            "response": response["response"].content,
            "retrieved_contexts": [doc.page_content for doc in response["context"]]
        })

        if name == "compression" or name == "ensemble":
            print(f"Sleeping for 7 seconds for {name}")
            time.sleep(7)  # 7 seconds to be safe

    results[name] = EvaluationDataset.from_list(eval_samples)
    print(f"Completed: {name}")



Completed: naive
Completed: bm25
Completed: parent_document
Sleeping for 7 seconds for compression
Sleeping for 7 seconds for compression
Sleeping for 7 seconds for compression
Sleeping for 7 seconds for compression
Sleeping for 7 seconds for compression
Sleeping for 7 seconds for compression
Sleeping for 7 seconds for compression
Sleeping for 7 seconds for compression
Sleeping for 7 seconds for compression
Sleeping for 7 seconds for compression
Sleeping for 7 seconds for compression
Sleeping for 7 seconds for compression
Completed: compression
Completed: multi_query
Sleeping for 7 seconds for ensemble
Sleeping for 7 seconds for ensemble
Sleeping for 7 seconds for ensemble
Sleeping for 7 seconds for ensemble
Sleeping for 7 seconds for ensemble
Sleeping for 7 seconds for ensemble
Sleeping for 7 seconds for ensemble
Sleeping for 7 seconds for ensemble
Sleeping for 7 seconds for ensemble
Sleeping for 7 seconds for ensemble
Sleeping for 7 seconds for ensemble
Sleeping for 7 seconds for ens

In [66]:
from ragas import evaluate
from ragas.run_config import RunConfig
from langchain_core.tracers import LangChainTracer
from ragas.metrics import LLMContextRecall, LLMContextPrecisionWithoutReference, ContextEntityRecall

evaluation_results = {}
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
retriever_metrics = [LLMContextRecall(), LLMContextPrecisionWithoutReference(), ContextEntityRecall()]

for name, eval_dataset in results.items():

    result = evaluate(
        dataset=eval_dataset,
        metrics=retriever_metrics,
        llm=evaluator_llm
    )
    evaluation_results[name] = result
    print(f"{name}: {result}")

/var/folders/vv/l_xmncsj59gbr7ptc7h1ghvm0000gn/T/ipykernel_67708/398805552.py:4: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, LLMContextPrecisionWithoutReference, ContextEntityRecall
/var/folders/vv/l_xmncsj59gbr7ptc7h1ghvm0000gn/T/ipykernel_67708/398805552.py:4: DeprecationWarning: Importing LLMContextPrecisionWithoutReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithoutReference
  from ragas.metrics import LLMContextRecall, LLMContextPrecisionWithoutReference, ContextEntityRecall
/var/folders/vv/l_xmncsj59gbr7ptc7h1ghvm0000gn/T/ipykernel_67708/398805552.py:4: DeprecationWarning: Importing ContextEntityRecall from 'ragas.metri

Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

naive: {'context_recall': 0.8333, 'llm_context_precision_without_reference': 0.9861, 'context_entity_recall': 0.3939}


Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

bm25: {'context_recall': 0.2778, 'llm_context_precision_without_reference': 0.2847, 'context_entity_recall': 0.2286}


...

compression: {'context_recall': 0.7708, 'llm_context_precision_without_reference': 0.9861, 'context_entity_recall': 0.3476}


Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

multi_query: {'context_recall': 0.8333, 'llm_context_precision_without_reference': 0.7847, 'context_entity_recall': 0.3421}


Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

ensemble: {'context_recall': 0.9792, 'llm_context_precision_without_reference': 0.6560, 'context_entity_recall': 0.4234}


## Retriever Comparison: Cost, Latency & Performance

| Retriever | Total Tokens | P50 Latency | Context Recall | Context Precision | Entity Recall |
|-----------|--------------|-------------|----------------|-------------------|---------------|
| **naive** | 18,845 | 1.27s | 0.833 | **0.986** | 0.394 |
| **bm25** | 7,147 | 1.29s | 0.278 | 0.285 | 0.229 |
| **parent_document** | 15,860 | 1.50s | 0.896 | 0.778 | **0.493** |
| **compression** | 7,784 | 1.62s | 0.771 | **0.986** | 0.348 |
| **multi_query** | 25,997 | 2.70s | 0.833 | 0.785 | 0.342 |
| **ensemble** | 42,342 | 3.66s | **0.979** | 0.656 | 0.423 |


### Activity Reflection

Based on the insights above, it seems like the ensemble retriever approach works best for this dataset without considering cost or latency, and compression is the best choice for good performance while balancing cost and latency.  Naive retrieval also performed well. Compression looks like it strikes a good balance between cost, recall and precision metrics, and latency. BM25 is NOT a good candidate, as it did very poorly in terms of metrics. Surprisingly, the naive approach did fairly well on retrieval metrics when compared with more powerful alternatives.
